In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import numpy as np
import healpy as hp

# Table IV

The goal of this notebook is to collect and print the event counts, energy, and angular resolution for each split-sky tier.
- Counts are extracted from the combined fits files
- Angular resolution is calculated from simulation
- Energy values are _not_ calculated in this notebook, but are taken from Table III

### Event Counts

In [3]:
map_dir = Path('/data/ana/CosmicRay/Anisotropy/IceTop/ITpass2/output/sidereal_unblinded')
it_split = np.radians(38.)

# Storage for data table
t = {f'{tier}{ab}':{} for tier in range(1,5) for ab in ['a','b']}

total = 0
for tier in range(1,5):
    
    tier_path = map_dir / f'tier{tier}/reconstruction/relintensityiter'
    map_files = tier_path.glob('combined*.fits.gz')
    map_file = sorted(map_files)[-1]

    data, _, _ = hp.read_map(map_file, range(3))

    npix = len(data)
    nside = hp.npix2nside(npix)
    theta, phi = hp.pix2ang(nside, range(npix))

    theta_cut = (theta >= np.pi - it_split)

    t[f'{tier}a']['counts'] = data[theta_cut].sum()
    t[f'{tier}b']['counts'] = data[~theta_cut].sum()
    
    print(f'{tier}a : {data[theta_cut].sum()/1e8:.2f}')
    print(f'{tier}b : {data[~theta_cut].sum()/1e8:.2f}')
    total += data.sum()

print(f'Total : {total/1e8:.2f}')

1a : 4.74
1b : 0.78
2a : 3.17
2b : 0.67
3a : 2.07
3b : 0.44
4a : 0.22
4b : 0.06
Total : 12.15


### Angular Resolution

In [4]:
# Define a function for weighted quantiles
def weighted_quantiles(values, weights, quantiles=[0.5]):
    if len(values) == 0:
        return 0
    else:
        i = np.argsort(values)
        c = np.cumsum(weights[i])
        return values[i[np.searchsorted(c, np.asarray(quantiles) * c[-1])]]


def find_opening_angle(true_zen, reco_zen, true_az, reco_az):

    cospsi = (
        np.cos(true_zen) * np.cos(reco_zen)
        + np.sin(true_zen) * np.sin(reco_zen)
        * np.cos(reco_az - true_az)
    )

    # Avoid nan's in rounding cases
    cospsi = np.clip(cospsi, -1.0, 1.0)

    return np.rad2deg(np.arccos(cospsi))

In [5]:
year = 2012
model = 'SIBYLL2.1'

# Load in the SIBYLL2.1 particle sims - note that these will also be your variable names
KEYS = ['hits', 'reco_pass', 'showerplane_zen', 'showerplane_az', 'laputop_zen', 'laputop_az', 'true_zenith', 'true_azimuth', 'Gweights']

for key in KEYS:
    sim_file = f'/data/user/tfutrell/it_anisotropy/{year}/{model}/{key}.npy'
    with open(sim_file, 'rb') as file:
        globals()[key] = np.load(file)

# Directional reconstruction cuts
plane_cut   = (showerplane_zen < np.radians(55))
laputop_cut = (laputop_zen < np.radians(55))
reco_cut    = (reco_pass == 1)

# Energy tier cuts
offset = np.ceil((year - 2011)/2)
TIERS = {
    'Tier 1': (3 <= hits) * (hits < 5),
    'Tier 2': (5 <= hits) * (hits < 10 - offset),
    'Tier 3': (10 - offset <= hits) * (hits < 17 - offset),
    'Tier 4': (17 - offset <= hits)
}

# Weights
it_weights = Gweights

In [6]:
for tier, tier_cut in TIERS.items():

    if tier == 'Tier 1':
        dphi = find_opening_angle(true_zenith, showerplane_zen, true_azimuth, showerplane_az)
        theta_cut = (showerplane_zen <= it_split)
        cut = tier_cut * plane_cut
    else:
        dphi = find_opening_angle(true_zenith, laputop_zen, true_azimuth, laputop_az)
        theta_cut = (laputop_zen <= it_split)
        cut = tier_cut * laputop_cut * reco_cut

    # Tier _a
    med, sigl, sigr = weighted_quantiles(dphi[cut * theta_cut], it_weights[cut * theta_cut], [0.5, 0.16, 0.84])
    label = f'{tier[-1]}a'
    t[label]['dphi'], t[label]['sigl'], t[label]['sigr'] = med, med-sigl, sigr-med

    # Tier _b
    med, sigl, sigr = weighted_quantiles(dphi[cut * ~theta_cut], it_weights[cut * ~theta_cut], [0.5, 0.16, 0.84])
    label = f'{tier[-1]}b'
    t[label]['dphi'], t[label]['sigl'], t[label]['sigr'] = med, med-sigl, sigr-med

### Table printing

In [7]:
energies = [0.27, 0.42, 0.82, 1.38, 2.14, 3.31, 6.01, 9.24]

for i, (tier, tier_dict) in enumerate(t.items()):
    print(f'{tier} & {energies[i]} & {tier_dict["counts"]/1e8:.2f} & ${tier_dict["dphi"]:.2f}_{{-{tier_dict["sigl"]:.2f}}}^{{+{tier_dict["sigr"]:.2f}}}$ \\\\[0.1cm]')

1a & 0.27 & 4.74 & $1.72_{-0.92}^{+1.89}$ \\[0.1cm]
1b & 0.42 & 0.78 & $1.87_{-1.02}^{+2.32}$ \\[0.1cm]
2a & 0.82 & 3.17 & $0.71_{-0.38}^{+0.65}$ \\[0.1cm]
2b & 1.38 & 0.67 & $0.75_{-0.40}^{+0.71}$ \\[0.1cm]
3a & 2.14 & 2.07 & $0.47_{-0.25}^{+0.38}$ \\[0.1cm]
3b & 3.31 & 0.44 & $0.51_{-0.27}^{+0.42}$ \\[0.1cm]
4a & 6.01 & 0.22 & $0.33_{-0.17}^{+0.26}$ \\[0.1cm]
4b & 9.24 & 0.06 & $0.36_{-0.19}^{+0.31}$ \\[0.1cm]
